# 03 - Spatial Mapping
zones.json validation and coordinate system walkthrough.

In [ ]:
import sys
sys.path.insert(0, "..")
import json
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

## Load zones.json

In [ ]:
with open("../data/layout/zones.json") as f:
    zones = json.load(f)
print(f"Canvas : {zones['canvas']}")
print(f"Machines: {len(zones['machines'])}")
print(f"Areas   : {list(zones['areas'].keys())}")
print(f"Aisles  : {zones['aisles']}")

## Coordinate system outline

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9.35))
ax.set_xlim(0, 30)
ax.set_ylim(0, 20)
ax.set_aspect("equal")
ax.set_facecolor("#E8E8E8")
vsm_colors = {"Alpha": "#AED6F1", "Beta": "#A9DFBF", "Gamma": "#F9E79F"}
for mid, mz in zones["machines"].items():
    color = vsm_colors.get(mz["vsm"], "#DDD")
    ax.add_patch(Rectangle((mz["x"], mz["y"]), mz["w"], mz["h"],
        facecolor=color, edgecolor="#333", linewidth=0.8, alpha=0.75))
    ax.text(mz["x"] + mz["w"]/2, mz["y"] + mz["h"]/2, mid,
        ha="center", va="center", fontsize=5, fontweight="bold")
for aisle in zones["aisles"]:
    ax.add_patch(Rectangle((aisle["x"], 0), aisle["w"], 20,
        facecolor="#AAAAAA", alpha=0.55))
plt.title("zones.json - Coordinate System (VSM colored)")
plt.tight_layout()

## ID consistency check

In [ ]:
import pandas as pd
metrics = pd.read_csv("../data/processed/mtbf_metrics.csv")
zone_ids   = set(zones["machines"].keys())
metric_ids = set(metrics["machine_id"].unique())
print("In zones not in metrics :", zone_ids   - metric_ids)
print("In metrics not in zones :", metric_ids - zone_ids)
print("All IDs match            :", zone_ids == metric_ids)

## Machine coverage per area

In [ ]:
import pandas as pd
machine_df = pd.DataFrame(zones["machines"]).T.reset_index().rename(columns={"index":"machine_id"})
print(machine_df.groupby(["vsm","area"]).size().unstack(fill_value=0))